# Convert ALOHA HDF5 recordings to LeRobot

This notebook runs inspection, explicit planning, image conversion, and source-aware validation for the configured ALOHA episode. Only the optional two-dataset merge is gated by `RUN_MERGE`.

In [ ]:
from pathlib import Path
from pprint import pprint

from leport import convert, create_plan, inspect, merge, validate
from leport.conversion.plan import save_plan
from leport.sources import EpisodeSelection

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Project root not found; start from the leport root or notebooks/")

source_path = project_root / "data/aloha/episode_0.hdf5"
output_root = project_root / "outputs"
plan_path = output_root / "aloha-image-demo-a.yaml"
target_root = output_root / "aloha-image-demo-a"

print("Configured source:", source_path)

## 1. Inspect the source

A matching filename is not sufficient. Inspection requires frame-addressable `action` and `observations/qpos`, restores numeric episode order, and reports decoded camera schema for both raw and JPEG storage.

In [ ]:
if not source_path.exists():
    raise FileNotFoundError(f"ALOHA source not found: {source_path}")
inspection = inspect(source_path, adapter="aloha")
pprint(inspection.to_dict())

## 2. Define explicit semantics

Confirm these values from the recording setup and inspection. LePort does not infer FPS, task text, robot type, action meaning, joint order, units, or target feature names. Empty episode IDs select every episode; use `EpisodeSelection(episode_ids=("episode_0", "episode_10"))` for a subset.

In [ ]:
episode_selection = EpisodeSelection()
fps = 50
task = "move the object"
robot_type = "aloha"
action_source = "action"
state_sources = ("observations/qpos", "observations/qvel")
image_sources = {
    "observations/images/cam_high": "observation.images.high",
    "observations/images/cam_left_wrist": "observation.images.left_wrist",
    "observations/images/cam_right_wrist": "observation.images.right_wrist",
}
action_dtype = "float32"
state_dtype = "float32"
use_videos = True

## 3. Create and save a plan

State selector order defines the concatenated `observation.state` layout. This example maps two cameras to individual image features; set `use_videos = True` before planning to create video features.

In [ ]:
plan = create_plan(
    source_path,
    target_root=target_root,
    repo_id="local/aloha-image-demo-a",
    robot_type=robot_type,
    fps=fps,
    task=task,
    action_source=action_source,
    action_dtype=action_dtype,
    state_sources=state_sources,
    state_dtype=state_dtype,
    image_sources=image_sources,
    use_videos=use_videos,
    adapter="aloha",
    selection=episode_selection,
)
pprint(plan.to_dict())
output_root.mkdir(parents=True, exist_ok=True)
if plan_path.exists():
    raise FileExistsError(f"Plan already exists: {plan_path}")
save_plan(plan, plan_path)

## 4. Convert

Conversion preflights every selected episode, streams only mapped fields, decodes only the current JPEG frame, and commits the target only after LeRobot reload validation succeeds.

In [ ]:
if target_root.exists():
    raise FileExistsError(f"Target already exists: {target_root}")
conversion_result = convert(plan_path)
pprint(conversion_result.validation.to_dict())

## 5. Validate against the source

Providing the reviewed plan compares episode counts and lengths, planned features, tasks, and decoded visual features with source expectations.

In [ ]:
validation_report = validate(target_root, plan=plan_path)
pprint(validation_report.to_dict())

## 6. Merge converted datasets (optional)

Set `RUN_MERGE = True` to convert `episode_1.hdf5` with the same schema, then merge its LeRobot dataset with the converted `episode_0.hdf5` dataset. Merge accepts converted datasets, never raw HDF5 files.

In [ ]:
RUN_MERGE = False

merge_result = None
if RUN_MERGE:
    second_source_path = project_root / "data/aloha/episode_1.hdf5"
    second_dataset = output_root / "aloha-image-demo-b"
    merged_target = output_root / "aloha-image-demo-merged"

    if not second_source_path.exists():
        raise FileNotFoundError(f"ALOHA source not found: {second_source_path}")
    second_plan = create_plan(
        second_source_path,
        target_root=second_dataset,
        repo_id="local/aloha-image-demo-b",
        robot_type=robot_type,
        fps=fps,
        task=task,
        action_source=action_source,
        action_dtype=action_dtype,
        state_sources=state_sources,
        state_dtype=state_dtype,
        image_sources=image_sources,
        use_videos=use_videos,
        adapter="aloha",
        selection=episode_selection,
    )

    if second_dataset.exists():
        raise FileExistsError(f"Target already exists: {second_dataset}")
    second_conversion_result = convert(second_plan)
    pprint(second_conversion_result.validation.to_dict())

    if merged_target.exists():
        raise FileExistsError(f"Merge target already exists: {merged_target}")
    merge_result = merge(
        (target_root, second_dataset),
        target_root=merged_target,
        repo_id="local/aloha-image-demo-merged",
    )
    pprint(merge_result.to_dict())
else:
    print("Optional merge skipped.")

## Equivalent CLI commands

Run the equivalent commands from the repository root through `uv run ...`:

```bash
uv run leport inspect data/aloha/episode_0.hdf5 --adapter aloha --json

uv run leport plan \
  --source data/aloha/episode_0.hdf5 \
  --output outputs/aloha-image-demo-a.yaml \
  --adapter aloha \
  --target outputs/aloha-image-demo-a \
  --repo-id local/aloha-image-demo-a \
  --robot-type aloha --fps 50 --task "move the object" \
  --action action --action-dtype float32 \
  --state observations/qpos \
  --state observations/qvel \
  --state-dtype float32 \
  --image observations/images/cam_high=observation.images.high \
  --image observations/images/cam_left_wrist=observation.images.left_wrist \
  --image observations/images/cam_right_wrist=observation.images.right_wrist \
  --no-videos

uv run leport convert --config outputs/aloha-image-demo-a.yaml --json
uv run leport validate outputs/aloha-image-demo-a \
  --config outputs/aloha-image-demo-a.yaml --json

# Optional: prepare episode_1 with the same schema before merging.
uv run leport plan \
  --source data/aloha/episode_1.hdf5 \
  --output outputs/aloha-image-demo-b.yaml \
  --adapter aloha \
  --target outputs/aloha-image-demo-b \
  --repo-id local/aloha-image-demo-b \
  --robot-type aloha --fps 50 --task "move the object" \
  --action action --action-dtype float32 \
  --state observations/qpos \
  --state observations/qvel \
  --state-dtype float32 \
  --image observations/images/cam_high=observation.images.high \
  --image observations/images/cam_left_wrist=observation.images.left_wrist \
  --image observations/images/cam_right_wrist=observation.images.right_wrist \
  --no-videos

uv run leport convert --config outputs/aloha-image-demo-b.yaml --json
uv run leport merge \
  outputs/aloha-image-demo-a \
  outputs/aloha-image-demo-b \
  --target outputs/aloha-image-demo-merged \
  --repo-id local/aloha-image-demo-merged \
  --json
```

> Before rerunning conversion or merge, review and handle existing plan and target paths explicitly. LePort does not overwrite completed datasets silently.